In [1]:
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk import pos_tag
from nltk.tokenize import word_tokenize
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from transformers import pipeline
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

2025-09-21 21:25:30.779925: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-21 21:25:30.790642: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758453930.804547  477033 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758453930.808979  477033 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758453930.820256  477033 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
import pandas as pd

# Load financial news dataset from CSV file
df = pd.read_csv('New AI spreadsheet - Sheet1.csv')
#df = pd.read_csv('nurse_emotion.csv')
# Drop rows where 'content' is NaN
#df.dropna(subset=['Headlines'], inplace=True)

# Display the first few rows of the dataset
df.head()

,patientId,age,gender,observationStart,observationEnd,nursingNote,medications,heartRate,spo2,temperature,...,waterIntakeMl,mealsSkipped,exerciseMinutes,bathroomVisits,behaviourTags,emotionTags,clinicalSummary,entitiesExtracted,baselineStats,state
0,P0001,80,Male,2025-06-01T00:00:00+00:00,2025-06-01T06:00:00+00:00,Patient remains stable with a heart rate of 65...,"Acetaminophen(Taken)(500mg), Furosemide(Taken)...",65 bpm,97%,36.5 °C,...,411,0,5,3,"cooperative, complaining, sleepy, inPain, tired",NaN,Patient remains stable with a heart rate of 65...,"{symptoms=, vitals=, procedures=, medications=}","{avgHeartRate=70.1, avgSpo2=97.4, avgSleepHour...",NaN
1,P0001,80,Male,2025-06-01T06:00:00+00:00,2025-06-01T12:00:00+00:00,"Patient is currently stable and alert, reporti...","Amlodipine(Taken)(10mg), Lisinopril(Taken)(20mg)",76 bpm,97%,36.8 °C,...,616,1,16,3,"cooperative, talkative, sleepy, skippedMeals, ...",NaN,Patient is stable with a heart rate of 76 bpm ...,"{symptoms=, vitals=, procedures=, medications=}","{avgHeartRate=75.7, avgSpo2=96.4, avgSleepHour...",NaN
2,P0001,80,Male,2025-06-01T12:00:00+00:00,2025-06-01T18:00:00+00:00,"Patient is currently stable and alert, reporti...","Metoprolol(Taken)(25mg), Lisinopril(Delayed)(1...",67 bpm,99%,36.8 °C,...,650,0,6,1,"sleepy, tired",NaN,Patient remains stable with vital signs within...,"{symptoms=, vitals=, procedures=, medications=}","{avgHeartRate=78.5, avgSpo2=97.2, avgSleepHour...",NaN
3,P0001,80,Male,2025-06-01T18:00:00+00:00,2025-06-02T00:00:00+00:00,Patient remains stable with a heart rate of 73...,"Metoprolol(Taken)(50mg), Lisinopril(Taken)(10mg)",73 bpm,99%,36.7 °C,...,356,0,10,3,"sleepy, talkative, cooperative, tired",NaN,Patient remains stable with a heart rate of 73...,"{symptoms=, vitals=, procedures=, medications=}","{avgHeartRate=84.6, avgSpo2=97.6, avgSleepHour...",NaN
4,P0001,80,Male,2025-06-02T00:00:00+00:00,2025-06-02T06:00:00+00:00,"Patient remains uncomfortable, but vital signs...","Oxygen(Delayed)(2L/min via nasal cannula), Pre...",85 bpm,92%,37.1 °C,...,254,1,4,3,"sleepy, skippedMeals, refusedActivity, noncomp...",frustrated,Patient remains uncomfortable with fatigue and...,"{symptoms=, vitals=, procedures=, medications=}","{avgHeartRate=74.6, avgSpo2=95.2, avgSleepHour...",NaN


In [3]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Text preprocessing function
def preprocess_text(text):
    text = re.sub(r"http\S+", "", text)       # Remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)   # Remove special characters
    text = text.lower()                       # Lowercase
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

df['Cleaned_Content'] = df['nursingNote'].apply(preprocess_text)
df['Cleaned_Content'].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/unix_david/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    patient remains stable heart rate bpm oxygen s...
1    patient currently stable alert reporting compl...
2    patient currently stable alert reporting acute...
3    patient remains stable heart rate bpm oxygen s...
4    patient remains uncomfortable vital signs stab...
Name: Cleaned_Content, dtype: object

In [4]:


# Extend the custom lexicon with additional domain-specific words
custom_lexicon = {
    # 🌿 Positive / Improvement
    'good': 1.0,
    'great': 1.5,
    'excellent': 1.5,
    'effective': 1.0,
    'successful': 1.5,
    'beneficial': 1.5,
    'stable': 1.0,
    'improving': 1.2,
    'improved': 1.0,
    'recovering': 1.0,
    'alert': 0.8,
    'cooperative': 0.7,
    'resting well': 1.0,
    'responding well': 1.2,
    'comfortable': 1.0,
    'independent': 1.0,
    'calm': 0.8,
    'oriented': 0.6,
    'no acute distress': 1.5,
    'pain well controlled': 1.2,
    'symptoms improving': 1.2,
    'vitals stable': 1.0,
    'resting comfortably': 1.0,
    'slept well': 1.0,
    'tolerating diet': 0.8,
    'ambulating independently': 1.2,
    'mobilising well': 1.0,
    'discharge planned': 0.9,
    'follow up arranged': 0.6,
    'responded to treatment': 1.2,
    'good appetite': 0.8,
    'wound healing well': 1.3,
    'cooperative with care': 0.7,
    'engaged in therapy': 1.0,
    'supportive family present': 0.8,
    'pain decreased': 1.0,
    'no complaints': 1.0,
    'patient reassured': 0.9,
    'ambulating with assistance': 0.8,
    'positive progress': 1.2,
    'comfortable at rest': 1.0,
    'participating in care': 0.7,
    'patient motivated': 1.1,
    'compliant with medication': 1.2,
    'followed instructions': 0.9,
    'encouraged and accepted': 0.8,
    'family supportive': 0.9,

    # ⚠️ Negative / Decline
    'bad': -1.0,
    'poor': -1.5,
    'concern': -0.5,
    'uncertainty': -1.0,
    'unusual': -0.5,
    'unprecedented': -0.5,
    "can't": -0.9,
    'behind': -0.1,
    'rough': -0.5,
    'out': -0.3,
    'critical': -1.5,
    'deteriorated': -1.2,
    'drowsy': -0.6,
    'agitated': -1.0,
    'confused': -0.9,
    'pain': -1.2,
    'severe pain': -1.5,
    'distressed': -1.2,
    'unstable': -1.3,
    'non-compliant': -0.8,
    'risk': -1.0,
    'infection': -1.0,
    'falls risk': -1.2,
    'bleeding': -1.3,
    'weakness': -0.7,
    'acute distress': -1.5,
    'complains of pain': -1.2,
    'pain not controlled': -1.3,
    'severe discomfort': -1.5,
    'shortness of breath': -1.2,
    'respiratory distress': -1.5,
    'non-compliant with care': -1.0,
    'refused medication': -1.2,
    'poor appetite': -0.8,
    'unable to mobilise': -1.0,
    'risk of fall': -1.2,
    'high risk': -1.3,
    'unsteady gait': -1.0,
    'deteriorating condition': -1.4,
    'requires assistance': -0.6,
    'not responding to treatment': -1.5,
    'confused and agitated': -1.3,
    'family concerned': -0.8,
    'infection suspected': -1.1,
    'wound not healing': -1.4,
    'bleeding noted': -1.3,
    'vomiting': -1.1,
    'nausea': -0.9,
    'loss of appetite': -1.0,
    'fainted': -1.3,
    'collapse': -1.5,
    'unresponsive': -1.5,
    'labored breathing': -1.3,
    'low oxygen': -1.2,
    'chest pain': -1.5,
    'high fever': -1.2,
    'diarrhea': -0.9,
    'bed sores': -1.0,
    'non-verbal and restless': -1.2,
    'anxious': -0.7,
    'crying': -0.9,
    'complained repeatedly': -1.1,
    'requires restraint': -1.3,
    'fall reported': -1.4,
    'adverse reaction': -1.2,
    'code blue': -1.5,
    'emergency transfer': -1.4,

    # 📋 Neutral / Observational
    'week': 0.0,
    'stay long': 1.0,
    'turn': 0.5,
    'turning point': 1.0,
    'coronavirus': 0.0,
    'record': 0.1,
    'latest': 0.1,
    'normal': 0.1,
    'despite': -0.1,
    'without': -0.1,
    'minimal': 0.1,
    'admitted': 0.0,
    'discharged': 0.0,
    'shift handover': 0.0,
    'vitals': 0.0,
    'blood pressure': 0.0,
    'heart rate': 0.0,
    'oxygen saturation': 0.0,
    'temperature': 0.0,
    'medication given': 0.0,
    'wound dressing': 0.0,
    'follow up': 0.0,
    'admitted to ward': 0.0,
    'discharged home': 0.0,
    'shift handover complete': 0.0,
    'vital signs monitored': 0.0,
    'medication administered': 0.0,
    'iv fluids running': 0.0,
    'oxygen therapy ongoing': 0.0,
    'physio attended': 0.0,
    'labs sent': 0.0,
    'awaiting results': 0.0,
    'procedure completed': 0.0,
    'follow up scheduled': 0.0,
    'review pending': 0.0,
    'patient education given': 0.0,
    'family updated': 0.0,
    'multidisciplinary meeting': 0.0,
    'x-ray done': 0.0,
    'ct scan performed': 0.0,
    'mri scheduled': 0.0,
    'ecg recorded': 0.0,
    'blood test ordered': 0.0,
    'awaiting specialist review': 0.0,
    'doctor reviewed': 0.0,
    'nurse in attendance': 0.0,
    'ward round completed': 0.0,
    'care plan updated': 0.0,
    'dietician referral': 0.0,
    'social worker present': 0.0,
    'palliative care consulted': 0.0,
    'spiritual care offered': 0.0,
    'referral made': 0.0,
}


# Function to extract adjectives and verbs
def extract_adj_verbs(text):
    words = word_tokenize(text)
    tagged_words = pos_tag(words)

    adjectives = [word for word, tag in tagged_words if tag in ('JJ', 'JJR', 'JJS')]
    verbs = [word for word, tag in tagged_words if tag.startswith('VB')]
    
    return adjectives, verbs
    
# Apply the function to extract adjectives and verbs
df['Adjectives'], df['Verbs'] = zip(*df['Cleaned_Content'].apply(extract_adj_verbs))

# Display the DataFrame with the new sentiment column
df[['nursingNote', 'Cleaned_Content', 'Adjectives', 'Verbs']]

,nursingNote,Cleaned_Content,Adjectives,Verbs
0,Patient remains stable with a heart rate of 65...,patient remains stable heart rate bpm oxygen s...,"[stable, administered, able, usual, ml]","[remains, tolerated, reporting, taken]"
1,"Patient is currently stable and alert, reporti...",patient currently stable alert reporting compl...,"[stable, vital, normal, amlodipine, able]","[pain, took, scheduled, prescribed, resting]"
2,"Patient is currently stable and alert, reporti...",patient currently stable alert reporting acute...,"[stable, acute, vital, normal, ml, light, mini...","[reporting, consumed, tolerated, administered,..."
3,Patient remains stable with a heart rate of 73...,patient remains stable heart rate bpm oxygen s...,"[stable, comfortable, able, complete, adequate]","[remains, took, scheduled, doses, evening, fee..."
4,"Patient remains uncomfortable, but vital signs...",patient remains uncomfortable vital signs stab...,"[uncomfortable, vital, supplemental, nasal, mm...","[remains, stabilized, elevated, c, administere..."
...,...,...,...,...
1915,Patient remains stable with a heart rate of 68...,patient remains stable heart rate bpm oxygen s...,"[stable, patient, comfortable, lisinopril, pat...","[remains, mmhg, feeling, administered, schedul..."
1916,Patient remains febrile at 38.4°C and exhibits...,patient remains febrile c exhibits elevated bl...,"[febrile, comfortable, small, scheduled]","[remains, elevated, tolerating, completed, mis..."
1917,"Patient is currently stable and alert, reporti...",patient currently stable alert reporting compl...,"[stable, vital, normal, full]","[pain, consumed, received, prescribed]"
1918,"Patient reports continued discomfort, but appe...",patient reports continued discomfort appears c...,"[discomfort, comfortable, supplemental, nasal,...","[continued, appears, administered, tolerated, ..."


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, pipeline

# ---------------------------
# 1) Use Bio_ClinicalBERT
# ---------------------------
MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "Negative", 1: "Positive"},
    label2id={"Negative": 0, "Positive": 1}
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, config=config)

# Sentiment pipeline (will work now; gets MUCH better once you fine-tune on your data)
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, truncation=True)


# 2) enhanced sentiment function stays the same

from nltk import word_tokenize, pos_tag

def enhanced_sentiment_analysis(text, adjectives, verbs, model_sentiment):
    words = word_tokenize(text)
    tagged_words = pos_tag(words)

    positive_score = 0.0
    negative_score = 0.0
    is_negation = False

    for word in words:
        w = word.lower()
        if w == "not":
            is_negation = True
            continue

        word_score = custom_lexicon.get(w, 0.0)
        if is_negation:
            word_score = -word_score
            is_negation = False

        if word_score > 0:
            positive_score += word_score
        elif word_score < 0:
            negative_score += abs(word_score)

    for adj in adjectives:
        if adj in custom_lexicon:
            s = custom_lexicon[adj]
            positive_score += s if s > 0 else 0
            negative_score += abs(s) if s < 0 else 0

    for verb in verbs:
        if verb in custom_lexicon:
            s = custom_lexicon[verb]
            positive_score += s if s > 0 else 0
            negative_score += abs(s) if s < 0 else 0

    # Combine model sentiment with lexicon cues
    final_sentiment = model_sentiment
    if positive_score > negative_score:
        final_sentiment = "Positive"
    elif negative_score > positive_score:
        final_sentiment = "Negative"

    return final_sentiment

#text_col = "nursingNote" if "nursingNote" in df.columns else "Cleaned_Content"
text_col = "Cleaned_Content"
# already had an extractor; leaving name the same
# df['Adjectives'], df['Verbs'] = zip(*df[text_col].apply(extract_adj_verbs))

# Model sentiment using Bio_ClinicalBERT
df['ClinicalBERT_Sentiment'] = df[text_col].apply(lambda x: nlp(str(x))[0]['label'])

# Enhanced fusion with  lexicon
df['Enhanced_NLTK_Sentiment'] = df.apply(
    lambda row: enhanced_sentiment_analysis(
        str(row[text_col]),
        row['Adjectives'],
        row['Verbs'],
        row['ClinicalBERT_Sentiment']
    ),
    axis=1
)

# Optional: if you still want to show the original columns
display_cols = [c for c in ['nursingNote', text_col, 'ClinicalBERT_Sentiment', 'Enhanced_NLTK_Sentiment'] if c in df.columns]
df[display_cols]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,nursingNote,Cleaned_Content,ClinicalBERT_Sentiment,Enhanced_NLTK_Sentiment
0,Patient remains stable with a heart rate of 65...,patient remains stable heart rate bpm oxygen s...,Positive,Positive
1,"Patient is currently stable and alert, reporti...",patient currently stable alert reporting compl...,Positive,Positive
2,"Patient is currently stable and alert, reporti...",patient currently stable alert reporting acute...,Positive,Positive
3,Patient remains stable with a heart rate of 73...,patient remains stable heart rate bpm oxygen s...,Positive,Positive
4,"Patient remains uncomfortable, but vital signs...",patient remains uncomfortable vital signs stab...,Positive,Positive
...,...,...,...,...
1915,Patient remains stable with a heart rate of 68...,patient remains stable heart rate bpm oxygen s...,Positive,Positive
1916,Patient remains febrile at 38.4°C and exhibits...,patient remains febrile c exhibits elevated bl...,Positive,Positive
1917,"Patient is currently stable and alert, reporti...",patient currently stable alert reporting compl...,Positive,Positive
1918,"Patient reports continued discomfort, but appe...",patient reports continued discomfort appears c...,Positive,Positive


# 🔁 Sentiment Analysis **Emotional Polarity Detection**  using Glove , transformer(llms)

This version swaps the  tone classifier for a general **emotion classifier** and derives **polarity** (Positive/Negative) from the predicted emotion.

**Outputs added:**
- `Emotion_Label` – single best emotion label per text
- `Polarity_3class` – `Positive`, `Negative`, or `Neutral`
- `Polarity_2class` – `Positive` or `Negative` (maps `Neutral` to `Negative` by default; adjust if you prefer)


In [6]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Download VADER lexicon
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

# Load pre-trained Word2Vec or GloVe model
# For example, load GloVe vectors (download from Stanford GloVe site if needed)
word_vectors = KeyedVectors.load_word2vec_format('glove.6B.100d.txt', binary=False, no_header=True)
# Replace 'path/to/glove.6B.100d.txt' with actual path to the GloVe file

# Extract words from VADER lexicon and filter those in embedding model
vader_lexicon = sia.lexicon
vader_words = {word: word_vectors[word] for word in vader_lexicon if word in word_vectors}

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/unix_david/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [7]:
## Extract words from VADER lexicon and filter those in embedding model
vader_lexicon = sia.lexicon
vader_words = {word: word_vectors[word] for word in vader_lexicon if word in word_vectors}

# Pre-calculate matrix of VADER word embeddings for efficient similarity calculations
vader_matrix = np.array(list(vader_words.values()))
vader_word_list = list(vader_words.keys())

# Caching dictionary for OOV words
oov_cache = {}

# Function to find the closest VADER word using cosine similarity
def get_closest_vader_score(word):
    if word in oov_cache:
        return oov_cache[word]

    if word in word_vectors:
        word_vector = word_vectors[word].reshape(1, -1)
        # Calculate cosine similarity for the word vector against all VADER words in one operation
        similarities = cosine_similarity(word_vector, vader_matrix).flatten()
        max_index = np.argmax(similarities)
        closest_word = vader_word_list[max_index]
        score = vader_lexicon[closest_word]
    else:
        score = 0  # Neutral score for words without embeddings

    # Cache result for the word to avoid re-calculation
    oov_cache[word] = score
    return score


# Find new sentiments for rows labeled as Neutral
neutral_rows = df[df['Enhanced_NLTK_Sentiment'] == 'Neutral'].index

for index in neutral_rows:
    words = df.loc[index, 'Cleaned_Content'].split()
    scores = [get_closest_vader_score(word) for word in words]
    avg_score = np.mean(scores)  # Average score of words

    # Classify based on score
    if avg_score >= 0.05:
        df.loc[index, 'Enhanced_NLTK_Sentiment'] = 'Positive'
    elif avg_score <= -0.05:
        df.loc[index, 'Enhanced_NLTK_Sentiment'] = 'Negative'
    else:
        df.loc[index, 'Enhanced_NLTK_Sentiment'] = 'Neutral'

In [8]:
# --- Step 1: Install Required Libraries ---
# Ensure you have the necessary libraries installed.
# The 'transformers' library requires PyTorch or TensorFlow.
# TextBlob is a new addition for this script.
try:
    import textblob
except ImportError:
    print("Installing textblob...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "textblob"])

# --- Step 2: Import All Necessary Libraries ---
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from transformers import pipeline, logging
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

# Suppress verbose warnings from the transformers library
logging.set_verbosity_error()




print(df[['nursingNote', 'Cleaned_Content']].head())


                                         nursingNote  \
0  Patient remains stable with a heart rate of 65...   
1  Patient is currently stable and alert, reporti...   
2  Patient is currently stable and alert, reporti...   
3  Patient remains stable with a heart rate of 73...   
4  Patient remains uncomfortable, but vital signs...   

                                     Cleaned_Content  
0  patient remains stable heart rate bpm oxygen s...  
1  patient currently stable alert reporting compl...  
2  patient currently stable alert reporting acute...  
3  patient remains stable heart rate bpm oxygen s...  
4  patient remains uncomfortable vital signs stab...  


In [9]:


# --- Initial Setup and Downloads ---

# Download necessary NLTK models
nltk.download('vader_lexicon')
nltk.download('punkt')

# 1. Initialize VADER (Your original method)
sia = SentimentIntensityAnalyzer()

# 2. Initialize a pre-trained LLM for sentiment analysis
# This uses the Hugging Face transformers library to download a model.
# The first time you run this, it will download the model (approx. 268MB).
print("Initializing LLM sentiment pipeline...")
llm_sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("LLM pipeline initialized.")

# 3. Load GloVe pre-trained vectors (as in your original code)
# IMPORTANT: You must download 'glove.6B.100d.txt' and place it in the same
# directory as this script. You can download it from the Stanford GloVe website.
try:
    print("Loading GloVe word vectors...")
    word_vectors = KeyedVectors.load_word2vec_format('glove.6B.100d.txt', binary=False, no_header=True)
    
    # Prepare VADER words and vectors for cosine similarity
    vader_lexicon = sia.lexicon
    vader_words_in_model = {word: word_vectors[word] for word in vader_lexicon if word in word_vectors}
    vader_matrix = np.array(list(vader_words_in_model.values()))
    vader_word_list = list(vader_words_in_model.keys())
    print("GloVe vectors loaded successfully.")
    glove_loaded = True
except FileNotFoundError:
    print("\nWARNING: 'glove.6B.100d.txt' not found.")
    print("The GloVe-enhanced VADER method will be skipped.")
    print("Please download it from the Stanford GloVe website to use this feature.\n")
    glove_loaded = False

# --- Sentiment Analysis Functions ---

# Function for GloVe-enhanced VADER to handle out-of-vocabulary words
def get_sentiment_with_glove(text):
    if not glove_loaded:
        return np.nan # Return Not a Number if GloVe model isn't loaded
        
    total_score = 0
    words = nltk.word_tokenize(text.lower())
    if not words:
        return 0
        
    for word in words:
        if word in sia.lexicon:
            total_score += sia.lexicon[word]
        elif word in word_vectors:
            # Word is not in VADER, but is in GloVe.
            # Find the most similar word in VADER's lexicon via cosine similarity.
            word_vector = word_vectors[word].reshape(1, -1)
            similarities = cosine_similarity(word_vector, vader_matrix).flatten()
            closest_word_index = np.argmax(similarities)
            closest_word = vader_word_list[closest_word_index]
            total_score += sia.lexicon[closest_word]
    
    return total_score / len(words) # Return an averaged score

# --- Create a Sample DataFrame ---

# Let's create some sample text to analyze
sample_data = {
    'text': [
        "The service was absolutely fantastic, I am so happy!",
        "I am utterly disappointed with the quality of this product.",
        "The movie was okay, but the ending was predictable.",
        "This is a revolutionary concept with groundbreaking features.",
        "The system is functional, though somewhat inefficient."
    ]
}
df = pd.DataFrame(sample_data)

# --- Apply All Sentiment Analysis Methods ---

print("\nRunning sentiment analysis...")

# Method 1: VADER (Original)
df['vader_sentiment'] = df['text'].apply(lambda t: sia.polarity_scores(t)['compound'])

# Method 2: TextBlob
df['textblob_polarity'] = df['text'].apply(lambda t: TextBlob(t).sentiment.polarity)
df['textblob_subjectivity'] = df['text'].apply(lambda t: TextBlob(t).sentiment.subjectivity)

# Method 3: LLM (Transformers)
# The pipeline returns a list with a dictionary, e.g., [{'label': 'POSITIVE', 'score': 0.99...}]
llm_results = llm_sentiment_pipeline(df['text'].tolist())
df['llm_label'] = [res['label'] for res in llm_results]
df['llm_score'] = [res['score'] for res in llm_results]

# Method 4: GloVe-Enhanced VADER
df['glove_vader_sentiment'] = df['text'].apply(get_sentiment_with_glove)

print("Analysis complete.\n")

# Display the final results
print("--- Comparison of Sentiment Analysis Methods ---")
print(df)

Initializing LLM sentiment pipeline...


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/unix_david/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /home/unix_david/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


LLM pipeline initialized.
Loading GloVe word vectors...
GloVe vectors loaded successfully.

Running sentiment analysis...
Analysis complete.

--- Comparison of Sentiment Analysis Methods ---
                                                text  vader_sentiment  \
0  The service was absolutely fantastic, I am so ...           0.8473   
1  I am utterly disappointed with the quality of ...          -0.5256   
2  The movie was okay, but the ending was predict...           0.1154   
3  This is a revolutionary concept with groundbre...           0.0000   
4  The system is functional, though somewhat inef...           0.0000   

   textblob_polarity  textblob_subjectivity llm_label  llm_score  \
0               0.70                   0.95  POSITIVE   0.999878   
1              -0.75                   0.75  NEGATIVE   0.999797   
2               0.15                   0.50  NEGATIVE   0.999376   
3               0.00                   0.00  POSITIVE   0.999876   
4               0.00          